In [6]:
# @title 1. Configuración del Entorno
# @markdown Este bloque instala librerías gráficas y carga módulos numéricos.

try:
    import schemdraw
except ImportError:
    print("Instalando librería de esquemáticos...")
    !pip install schemdraw > /dev/null
    import schemdraw

import schemdraw.elements as elm
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

print("✅ Entorno listo.")

✅ Entorno listo.


In [7]:
# @title 2. Base de Datos (OpAmp Worst-Case y Resistores)
# @markdown En este bloque se ubican los Parámetros actualizados del Amplificador Operacional a utilizar (LM324) en sus valores MÁXIMOS (Peor Caso) según Datasheet TI.

# Especificaciones del LM324 Estándar - Peor Caso @ 25°C
# Referencia: Datasheet TI
LM324_SPECS = {
    "GBW": 1.0e6,       # Gain Bandwidth Product (Usamos 1MHz conservador, datasheet dice 1.2MHz typ)
    "SR": 0.3e6,        # Slew Rate (Usamos 0.4 V/us conservador, datasheet dice 0.5 typ)
    "V_OS": 7e-3,       # Input Offset Voltage MAX (7mV)
    "V_OS_DRIFT": 20e-6,# Drift térmico estimado (Typ es 7uV/Cº, Max no especificado, usamos 20uV conservador)
    "I_B": 60e-9,      # Input Bias Current MAX (60nA)
    "I_OS": 5e-9,      # Input Offset Current MAX (5nA)
    "I_OS_DRIFT": 40e-12, # Drift corriente estimado (Typ es 10pA/Cº, Max no especificado, usamos 40pA/Cº conservador)
    "V_OUT_HIGH": 1.75,  # Vcc -1.75V drop (Worst case carga alta, Iout=-5mA)
    "V_OUT_LOW": 1.00   # Vss +1V drop (Worst case carga alta, Iout=1mA)
}

# Generación de Serie E24 estandar
E24_BASE = np.array([1.0, 1.1, 1.2, 1.3, 1.5, 1.6, 1.8, 2.0, 2.2, 2.4, 2.7, 3.0,
                     3.3, 3.6, 3.9, 4.3, 4.7, 5.1, 5.6, 6.2, 6.8, 7.5, 8.2, 9.1])
MULTIPLIERS = np.array([100, 1e3, 10e3, 100e3, 1e6])
RESISTORS = np.unique([round(v * m, 1) for v in E24_BASE for m in MULTIPLIERS])

print(f"✅ Base de datos (Worst-Case) cargada.")

✅ Base de datos (Worst-Case) cargada.


In [8]:
# @title 3. Motor de Cálculo
# @markdown Este bloque contiene los calculos utilizados para obtener los resultados

def calcular_circuitos_ponderados(gain1, gain2, vin_mv, f_op, vcc, vss, temp_op, peso_slider, config_type):
    """
    gain1: Ganancia principal (o rama Inversora en Diferencial)
    gain2: Ganancia secundaria (solo para Sumador)
    """
    resultados = []

    vin_max = vin_mv / 1000.0
    delta_t = abs(temp_op - 25.0)

    # Ajuste de parámetros
    vos_actual = LM324_SPECS["V_OS"] + (LM324_SPECS["V_OS_DRIFT"] * delta_t)
    ios_actual = LM324_SPECS["I_OS"] + (LM324_SPECS["I_OS_DRIFT"] * delta_t)

    # --- ESTRATEGIA ---
    # Iteramos sobre la Resistencia de Feedback (R2 o Rf) porque es el componente común.
    # Esto facilita encontrar Rin1 y Rin2 independientes.
    resistencias_fb = RESISTORS[RESISTORS >= 1000]

    for r_fb in resistencias_fb:

        # Inicializar valores
        r_in1 = 0
        r_in2 = 0 # Solo para sumador/diferencial
        ganancia_real_1 = 0
        ganancia_real_2 = 0
        noise_gain = 0

        # --- 1. CALCULAR RESISTENCIAS DE ENTRADA ---

        if config_type == 'No Inversor':
            # Av = 1 + Rf/Rin
            if gain1 <= 1: continue
            r_in_ideal = r_fb / (gain1 - 1)
            idx = (np.abs(RESISTORS - r_in_ideal)).argmin()
            r_in1 = RESISTORS[idx]
            ganancia_real_1 = 1 + (r_fb / r_in1)
            noise_gain = ganancia_real_1

        elif config_type == 'Inversor':
            # |Av| = Rf/Rin
            r_in_ideal = r_fb / gain1
            idx = (np.abs(RESISTORS - r_in_ideal)).argmin()
            r_in1 = RESISTORS[idx]
            ganancia_real_1 = r_fb / r_in1
            noise_gain = 1 + (r_fb / r_in1)

        elif config_type == 'Sumador Inversor (Ponderado)':
            # Canal 1
            r_in1_ideal = r_fb / gain1
            id1 = (np.abs(RESISTORS - r_in1_ideal)).argmin()
            r_in1 = RESISTORS[id1]
            ganancia_real_1 = r_fb / r_in1

            # Canal 2
            r_in2_ideal = r_fb / gain2
            id2 = (np.abs(RESISTORS - r_in2_ideal)).argmin()
            r_in2 = RESISTORS[id2]
            ganancia_real_2 = r_fb / r_in2

            # Noise Gain = 1 + Rf / (Rin1 || Rin2)
            r_parallel = (r_in1 * r_in2) / (r_in1 + r_in2)
            noise_gain = 1 + (r_fb / r_parallel)

        elif config_type == 'Diferencial':
            # Asumimos R_fb = R2 y R_in = R1 para mantener simetría y buen CMRR
            # Ad = R_fb / R_in
            r_in_ideal = r_fb / gain1
            idx = (np.abs(RESISTORS - r_in_ideal)).argmin()
            r_in1 = RESISTORS[idx]
            r_in2 = RESISTORS[idx] # Simetría

            ganancia_real_1 = r_fb / r_in1
            noise_gain = 1 + (r_fb / r_in1)

        # --- 2. FILTRO DE ERROR DE GANANCIA ---
        # Verificamos que la ganancia principal no se desvie mucho
        err_g1 = abs(ganancia_real_1 - gain1) / gain1 * 100
        if err_g1 > 15.0: continue

        if config_type == 'Sumador Inversor (Ponderado)':
             err_g2 = abs(ganancia_real_2 - gain2) / gain2 * 100
             if err_g2 > 15.0: continue
             # Promedio de errores para el score
             error_ganancia_pct = (err_g1 + err_g2) / 2
        else:
             error_ganancia_pct = err_g1

        # --- 3. VALIDACIONES DINÁMICAS ---

        # Ancho de Banda (Depende de Noise Gain)
        f_corte = LM324_SPECS["GBW"] / noise_gain
        if f_corte < f_op: continue

        # Voltaje de Salida Pico (Peor Caso: todas las entradas en fase al máximo)
        if config_type == 'Sumador Inversor (Ponderado)':
             vout_peak = (vin_max * ganancia_real_1) + (vin_max * ganancia_real_2)
        else:
             vout_peak = vin_max * ganancia_real_1

        # Slew Rate
        f_max_sr = LM324_SPECS["SR"] / (2 * np.pi * vout_peak)
        if f_max_sr < f_op: continue

        # Saturación
        v_sat_h = vcc - LM324_SPECS["V_OUT_HIGH"]
        if vout_peak > v_sat_h: continue

        # --- 4. CÁLCULO DE VARIABLES OPTIMIZACIÓN ---

        # Error DC (mV) = Vos * NoiseGain + Ios * Rf
        error_vos_out = vos_actual * noise_gain
        error_ios_out = ios_actual * r_fb
        error_total_mv = (error_vos_out + error_ios_out) * 1000

        # Consumo (uA) en feedback
        # Estimación simple: corriente máxima saliendo por Vout hacia el lazo
        if config_type == 'No Inversor':
            i_fb_ua = (vout_peak / (r_in1 + r_fb)) * 1e6
        elif config_type == 'Diferencial':
             i_fb_ua = (vout_peak / (r_in1 + r_fb)) * 1e6
        else: # Inversor y Sumador
             # Vout ve principalmente a Rf conectado a tierra virtual
             i_fb_ua = (vout_peak / r_fb) * 1e6

        resultados.append({
            "R_in1": r_in1,
            "R_in2": r_in2, # 0 si no se usa
            "R_fb": r_fb,
            "Av_Real": ganancia_real_1,
            "Av2_Real": ganancia_real_2,
            "Error_Ganancia_Pct": error_ganancia_pct,
            "Error_DC_mV": error_total_mv,
            "I_fb_uA": i_fb_ua,
            "BW_kHz": f_corte/1000
        })

    df = pd.DataFrame(resultados)
    if df.empty: return df

    # --- 5. PONDERACIÓN ---
    min_dc, max_dc = df["Error_DC_mV"].min(), df["Error_DC_mV"].max()
    min_gain, max_gain = df["Error_Ganancia_Pct"].min(), df["Error_Ganancia_Pct"].max()

    den_dc = (max_dc - min_dc) if (max_dc - min_dc) > 0 else 1.0
    den_gain = (max_gain - min_gain) if (max_gain - min_gain) > 0 else 1.0

    peso_gain = peso_slider
    peso_dc = 1.0 - peso_slider

    df["Norm_DC"] = (df["Error_DC_mV"] - min_dc) / den_dc
    df["Norm_Gain"] = (df["Error_Ganancia_Pct"] - min_gain) / den_gain

    df["Score"] = (peso_dc * df["Norm_DC"]) + (peso_gain * df["Norm_Gain"])

    return df.sort_values(by="Score", ascending=True).head(3)

print("✅ Motor de calculos listo.")

✅ Motor de calculos listo.


In [23]:
# @title 4. Interfaz de Usuario (GUI)
# @markdown Este bloque presenta los detalles sobre la salida de datos, generacion de tablas, vista y seleccion del circuito. La interfaz de usuario en general
style = {'description_width': '100px'}
layout_full = widgets.Layout(width='98%')
layout_half = widgets.Layout(width='48%')

# Widget de Configuración
w_conf = widgets.Dropdown(
    options=['No Inversor', 'Inversor', 'Sumador Inversor (Ponderado)', 'Diferencial'],
    value='No Inversor',
    description='Configuración:',
    style=style, layout=layout_full
)

w_vcc = widgets.FloatText(value=15.0, description='Vcc (+V):', style=style, layout=layout_half)
w_vss = widgets.FloatText(value=-15.0, description='Vss (-V):', style=style, layout=layout_half)

# GANANCIAS
w_gain1 = widgets.FloatText(value=10.0, description='Ganancia 1:', style=style, layout=layout_half)
w_gain2 = widgets.FloatText(value=5.0, description='Ganancia 2 (Sum):', style=style, layout=layout_half)

w_vin = widgets.FloatText(value=100.0, description='Vin Pico (mV):', style=style, layout=layout_half)
w_freq = widgets.FloatText(value=1000.0, description='Frecuencia (Hz):', style=style, layout=layout_half)
w_temp = widgets.FloatSlider(value=25.0, min=-40, max=85, step=5, description='Temp. (°C):', style=style, layout=layout_full)

w_peso = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.25,
    description='Prioridad:',
    readout_format='.2f',
    style=style, layout=layout_full
)
lbl_peso = widgets.Label(value="⬅ Menor Error DC . . . . . . . . . . . Menor Error Ganancia ➡")

btn_calc = widgets.Button(
    description=" CALCULAR Y DIBUJAR ",
    button_style='success',
    layout=widgets.Layout(width='100%', height='50px'),
    icon='table'
)

out_display = widgets.Output()

def draw_schematic(config, row, vin_val, g1_target, g2_target):
    d = schemdraw.Drawing()
    d += (op := elm.Opamp(leads=True).label('LM324'))

    r_fb = int(row['R_fb'])
    r_in1 = int(row['R_in1'])

    if config == 'No Inversor':
        d += elm.Line().left(d.unit/4).at(op.in2)
        d += elm.SourceSin().down().label('Vin')
        d += elm.Ground()
        d += elm.Line().left(d.unit).at(op.in1)
        d += (node_inv := elm.Dot())
        d += elm.Resistor().down().label(f'R1')
        d += elm.Ground()
        d += elm.Line().up(d.unit*1).at(node_inv.start)
        d += elm.Resistor().right().label(f'R2').length(d.unit*2.5)
        d += elm.Line().down().toy(op.out)
        d += elm.Line().left().tox(op.out)

    elif config == 'Inversor':
        d += elm.Line().left(d.unit/4).at(op.in1)
        d += (node_inv := elm.Dot())
        d += elm.Resistor().left().label(f'R1')
        d += elm.SourceSin().down().label('Vin')
        d += elm.Ground()
        d += elm.Line().down(d.unit/2).at(op.in2)
        d += elm.Ground()
        d += elm.Line().up(d.unit*1.5).at(node_inv.start)
        d += elm.Resistor().right().label(f'R2').length(d.unit*2.5)
        d += elm.Line().down().toy(op.out)
        d += elm.Line().left().tox(op.out)

    elif config == 'Sumador Inversor (Ponderado)':
        r_in2 = int(row['R_in2'])
        d += elm.Line().left(d.unit/4).at(op.in1)
        d += (node_sum := elm.Dot())

        # Rama 1
        d += elm.Line().up(3).at(node_sum.start)
        d += elm.Resistor().left().label(f'R1a')
        d += elm.SourceSin().down(d.unit/2).label('V1')
        d += elm.Ground()

        # Rama 2
        d += elm.Line().down(0.01).at(node_sum.start)
        d += elm.Resistor().left().label(f'R1b')
        d += elm.SourceSin().down(d.unit/2).label('V2')
        d += elm.Ground()

        d += elm.Line().down(d.unit/4).at(op.in2)
        d += elm.Ground()

        d += elm.Line().up(d.unit*2).at(node_sum.start)
        d += elm.Resistor().right().label(f'R2').length(d.unit*2.5)
        d += elm.Line().down().toy(op.out)
        d += elm.Line().left().tox(op.out)

    elif config == 'Diferencial':
        d += elm.Line().left(d.unit/4).at(op.in1)
        d += (node_inv := elm.Dot())
        d += elm.Resistor().left(d.unit*2).label(f'R1')
        d += elm.SourceSin().down(d.unit/2).label('V1')
        d += elm.Ground()
        d += elm.Line().up(d.unit*1.5).at(node_inv.start)
        d += elm.Resistor().right().label(f'R2').length(d.unit*2.5)
        d += elm.Line().down().toy(op.out)
        d += elm.Line().left().tox(op.out)

        d += elm.Line().left(d.unit/4).at(op.in2)
        d += (node_non := elm.Dot())
        d += elm.Resistor().left().label(f'R3')
        d += elm.SourceSin().down(d.unit/2).label('V2')
        d += elm.Ground()
        d += elm.Resistor().down().at(node_non.start).label(f'R4')
        d += elm.Ground()

    d += elm.Line().up(d.unit*0.01).at(op.out)
    d += elm.Dot().label('Vout', loc='top')

    d.push()
    d += elm.Gap().at(op.center).down(d.unit).length(3)

    if config == 'No Inversor':
        txt = f"|Av| = {g1_target}"

    elif config == 'Inversor':
        txt = f"Av = {g1_target}"

    elif config == 'Diferencial':
        txt = f"Ad = {g1_target}"

    else: # Sumador Inversor
        txt = f"Vout = -({g1_target}*V1 + {g2_target}*V2)"

    d += elm.Label().label(f"{config}\n{txt}").color('blue')
    d.pop()

    display(d.draw())

def on_click_process(b):
    with out_display:
        clear_output()

        best_df = calcular_circuitos_ponderados(
            w_gain1.value, w_gain2.value, w_vin.value, w_freq.value,
            w_vcc.value, w_vss.value, w_temp.value, w_peso.value, w_conf.value
        )

        if best_df.empty:
            print("⚠️ No se encontró solución válida.")
            return

        p_gain = int(w_peso.value * 100)
        p_dc = int((1.0 - w_peso.value) * 100)

        print(f"RESULTADOS: {w_conf.value} (Ponderación: {p_dc}% Menos Error DC / {p_gain}% Menos Error Ganancia)")

        # --- IMPRESIÓN DE TABLAS ---
        if w_conf.value == 'Sumador Inversor (Ponderado)':
             # Tabla Específica para Sumador (Más ancha por las dos ganancias)
             print("=" * 115)
             header = f"{'#':<3} {'R1a (Ω)':<9} {'R1b (Ω)':<9} {'Rf (Ω)':<9} {'Err. G1':<18} {'Err. G2':<18} {'Error DC':<12} {'Consumo':<12}"
             print(header)
             print("-" * 115)

             rank = 1
             for idx, row in best_df.iterrows():
                prefix = "#1🏆" if rank == 1 else f"#{rank}"

                r_in1 = f"{int(row['R_in1'])}"
                r_in2 = f"{int(row['R_in2'])}"
                r_fb = f"{int(row['R_fb'])}"

                # Cálculo de errores individuales para mostrar
                err1_abs = abs(row['Av_Real'] - w_gain1.value)
                err1_pct = (err1_abs / w_gain1.value) * 100
                err1_str = f"{err1_abs:.2f} ({err1_pct:.1f}%)"

                err2_abs = abs(row['Av2_Real'] - w_gain2.value)
                err2_pct = (err2_abs / w_gain2.value) * 100
                err2_str = f"{err2_abs:.2f} ({err2_pct:.1f}%)"

                err_dc = f"{row['Error_DC_mV']:.1f} mV"
                cons = f"{row['I_fb_uA']:.1f} uA"

                print(f"{prefix:<3} {r_in1:<9} {r_in2:<9} {r_fb:<9} {err1_str:<18} {err2_str:<18} {err_dc:<12} {cons:<12}")
                rank += 1
             print("=" * 115)

        else:
             # Tabla Estándar para No Inv / Inv / Diferencial
             label_gain = "Av Real"
             if "Inversor" in w_conf.value: label_gain = "|Av| Real"
             if w_conf.value == 'Diferencial': label_gain = "Ad Real"

             print("=" * 105)
             header = f"{'#':<3} {'R1 (Ω)':<10} {'R2 (Ω)':<10} {label_gain:<10} {'Err. Ganancia':<22} {'Error DC':<12} {'Consumo':<12}"
             print(header)
             print("-" * 105)

             rank = 1
             for idx, row in best_df.iterrows():
                prefix = "#1🏆" if rank == 1 else f"#{rank}"

                r_in1 = f"{int(row['R_in1'])}"
                r_fb = f"{int(row['R_fb'])}"
                av_real = f"{row['Av_Real']:.3f}"

                # Error Ganancia con formato "Abs (Pct%)"
                target = w_gain1.value
                err_abs = abs(row['Av_Real'] - target)
                err_pct = row['Error_Ganancia_Pct']
                err_str = f"{err_abs:.3f} ({err_pct:.2f}%)"

                err_dc = f"{row['Error_DC_mV']:.1f} mV"
                cons = f"{row['I_fb_uA']:.1f} uA"

                print(f"{prefix:<3} {r_in1:<10} {r_fb:<10} {av_real:<10} {err_str:<22} {err_dc:<12} {cons:<12}")
                rank += 1
             print("=" * 105)

        print("\n")
        draw_schematic(w_conf.value, best_df.iloc[0], 0, w_gain1.value, w_gain2.value)

btn_calc.on_click(on_click_process)

# Layout
ui = widgets.VBox([
    widgets.HTML("<h3>🎛️ Optimizador LM324 (Multi-Topología)</h3>"),
    w_conf,
    widgets.HBox([w_gain1, w_gain2]),
    widgets.HBox([w_vcc, w_vss]),
    widgets.HBox([w_vin, w_freq]),
    w_temp,
    widgets.HTML("<hr>"),
    w_peso,
    lbl_peso,
    widgets.HTML("<br>"),
    btn_calc
])

display(ui)
display(out_display)

Output()